# Modelo Bag-of-Words (BoW): de texto a vectores de conteo

**Curso:** Procesamiento de Lenguaje Natural · Unidad 2 — Semántica distribucional y representación de texto  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 6 — Vector Semantics  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook muestra cómo convertir textos en números con el modelo **Bag-of-Words (bolsa de palabras)**: cada documento se representa como un vector que cuenta cuántas veces aparece cada palabra de un vocabulario fijo, sin importar su orden. Primero se construye el vocabulario a mano con el tokenizador de NLTK para ver qué ocurre por dentro, y luego con `CountVectorizer` de scikit-learn, la herramienta que se usa en la práctica. El resultado es una matriz documento–término que cualquier algoritmo de aprendizaje automático puede recibir como entrada.

**Objetivo:** resolver el primer problema de todo sistema de PLN clásico: los algoritmos solo operan con números, así que hay que transformar las reseñas en vectores, y entender las propiedades de esa representación (vocabulario, dispersión, pérdida del orden) que motivan técnicas posteriores como TF-IDF y los *embeddings*.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Finanzas y servicio al cliente — Enrutamiento de PQRS bancarias

- **Problema:** un banco recibe miles de peticiones, quejas y reclamos por correo en una bandeja única y los reparte a mano.
- **Entrada:** el texto de cada correo.
- **Cómo se aplica Bag-of-Words:**
  1. Se tokeniza cada correo y se construye el vocabulario con todos los correos históricos.
  2. Cada correo se convierte en un vector de conteos (`tarjeta`=2, `bloqueo`=1, `cuota`=0…) con `CountVectorizer`.
  3. Un clasificador lineal entrenado con correos ya repartidos aprende qué palabras indican cada área (tarjetas, créditos, fraude).
- **Salida:** área destino y su probabilidad.
- **Decisión:** si la probabilidad supera 0.8 el correo se enruta solo; si no, lo revisa una persona.

### Turismo y hospitalidad — Temas en reseñas de hoteles

- **Problema:** una cadena hotelera quiere saber de qué se quejan o qué elogian los huéspedes en cada sede.
- **Entrada:** reseñas de Booking y Tripadvisor con la sede a la que pertenecen.
- **Cómo se aplica Bag-of-Words:**
  1. Se quitan stopwords y se construye la matriz documento–término de todas las reseñas.
  2. Se suman los vectores de conteo de las reseñas de cada sede.
  3. Las columnas con más conteo por sede (`limpieza`, `ruido`, `desayuno`) muestran los temas dominantes.
- **Salida:** tabla tema × sede con la frecuencia de cada término.
- **Decisión:** plan de mejora por sede, empezando por los temas negativos más mencionados.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install pandas scikit-learn nltk
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `pathlib.Path` | Rutas de archivos independientes del sistema operativo |
| `pandas` | Carga y visualización de los datos en tablas |
| `sklearn.feature_extraction.text.CountVectorizer` | Construcción del vocabulario y de la matriz BoW |
| `nltk.tokenize.TweetTokenizer` | Tokenización de textos cortos e informales |

**Datos:** `data/dataset_reseñas_espanol.csv` (incluido en el repositorio).

## Marco metodológico: pipeline ML / LLM

![Pipeline ML y LLM](img/Pipeline_ML_LLM_4x3.png)

El diagrama tiene una franja común de datos (arriba), dos rutas de modelado y una franja de operación (abajo): **ruta clásica** (*feature-based ML*) y **ruta LLM** (*foundation models*). Este notebook recorre la **ruta clásica**. Etapas del diagrama que cubre:

| Etapa del diagrama | Dónde se aborda en este notebook |
|--------------------|----------------------------------|
| Business & Data Understanding | Contexto y propósito; sección 2 (exploración del conjunto de datos) |
| Data | Sección 2: carga de `dataset_reseñas_espanol.csv` |
| Data Preprocessing | Sección 3: tokenización y normalización a minúsculas con `TweetTokenizer` |
| Feature Engineering | Secciones 4 y 5: matriz documento–término con `CountVectorizer` |

Este notebook se detiene en la **ingeniería de características**: produce la entrada que usarán los modelos de la ruta clásica (Naive Bayes, regresión logística, SVM) en los notebooks siguientes. No entrena ni despliega un modelo.

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | BoW es una representación **transparente**: cada columna es una palabra concreta y cada valor un conteo verificable. Las secciones 4 y 5 muestran la matriz completa para que se pueda inspeccionar. |
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | La sección 2 revisa la composición del conjunto de datos (balance de clases y reseñas repetidas) antes de usarlo, porque un vocabulario construido con datos sesgados o duplicados traslada ese sesgo al modelo. |
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Se documenta un riesgo propio de la técnica: BoW pierde el orden y la negación (*“no es buena”* y *“es buena”* comparten casi todos sus términos). |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | Todas las herramientas usadas (NLTK, scikit-learn, pandas) son de código abierto, y el conjunto de datos está en el repositorio, lo que permite reproducir y estudiar el experimento completo. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Explicar la representación **Bag-of-Words** (bolsa de palabras) y sus supuestos principales
2. Construir un vocabulario manualmente con el **TweetTokenizer** de NLTK
3. Construir una matriz de características BoW con el **CountVectorizer** de scikit-learn
4. Interpretar una matriz documento–término y razonar sobre su dispersión (*sparsity*)
5. Identificar las limitaciones de BoW como motivación para TF-IDF y los *word embeddings*

## ¿Qué es el modelo Bag-of-Words?

BoW representa un documento como un **multiconjunto no ordenado de ocurrencias de palabras**: un vector de conteos de palabras sobre un vocabulario fijo. Se descartan el orden de las palabras y la gramática.

Dado el vocabulario V = {w₁, w₂, …, wₙ}, el documento dᵢ se convierte en:

```
dᵢ = [count(w₁, dᵢ),  count(w₂, dᵢ),  …,  count(wₙ, dᵢ)]
```

Esto produce una **matriz documento–término** de dimensión (|D| × |V|), normalmente muy dispersa.

## Conjunto de datos

**`dataset_reseñas_espanol.csv`** — 200 reseñas cortas de películas en español (`Reseña`) etiquetadas como *Positiva* / *Negativa*. Ubicado en `data/`.

---

## 1. Configuración del entorno


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import TweetTokenizer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

DIR_DATA = Path.cwd() / 'data'
print(f'Directorio de datos: {DIR_DATA}')
print(f'¿Existe el directorio?: {DIR_DATA.exists()}')

Directorio de datos: C:\Users\epuerta\OneDrive - Universidad Tecnológica de Bolívar\Apps\courseNLP\examples\data
¿Existe el directorio?: True


> **Nota:** `pathlib.Path.cwd()` resuelve las rutas relativas al directorio de trabajo del notebook, sin nombres de usuario fijos ni separadores propios de cada sistema operativo. No es necesario manipular `sys.path`; los paquetes instalados siempre se pueden importar.


---

## 2. Cargar el conjunto de datos


In [4]:
# Lee el CSV de reseñas: columnas 'Reseña' (texto) y 'Sentimiento' (etiqueta)
corpus = pd.read_csv(DIR_DATA / 'dataset_reseñas_espanol.csv', sep=',')

In [5]:
# Tipos de datos, número de filas y valores nulos por columna
corpus.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Reseña       200 non-null    str  
 1   Sentimiento  200 non-null    str  
dtypes: str(2)
memory usage: 12.8 KB


In [6]:
# Primeras 10 reseñas para revisar el formato del texto
corpus.head(10)

,Reseña,Sentimiento
0,Muy entretenida y emotiva.,Positiva
1,Pésimas actuaciones y un ritmo lento.,Negativa
2,Gran dirección y un guion muy sólido.,Positiva
3,Pésimas actuaciones y un ritmo lento.,Negativa
4,Una historia conmovedora con actuaciones autén...,Positiva
5,Una completa pérdida de tiempo.,Negativa
6,"Me encantó cada escena, muy recomendada.",Positiva
7,Una trama aburrida y predecible.,Negativa
8,"Me encantó cada escena, muy recomendada.",Positiva
9,La película carece de coherencia narrativa.,Negativa


> **Interpretación de la salida:** El conjunto de datos tiene 200 filas y 2 columnas: `Reseña` (texto de la reseña) y `Sentimiento` (etiqueta: *Positiva* / *Negativa*). Ambas son columnas `object` (cadenas) sin valores nulos. Las clases están balanceadas (100 / 100).
>
> **Atención:** las 200 filas corresponden a solo **20 reseñas distintas**, repetidas entre 5 y 16 veces cada una (la celda siguiente lo verifica). Para ilustrar BoW no importa, pero para entrenar un clasificador provocaría **fuga de datos**: la misma reseña quedaría en entrenamiento y en prueba.

In [7]:
# Verifica cuántas reseñas son distintas y cuántas veces se repite cada una
print(f'Reseñas distintas: {corpus["Reseña"].nunique()} de {len(corpus)}')
corpus['Reseña'].value_counts().describe()[['min', 'max', 'mean']].round(1)

Reseñas distintas: 20 de 200


min      5.0
max     16.0
mean    10.0
Name: count, dtype: float64

---

## 3. Vocabulario manual con TweetTokenizer

Construimos un vocabulario manualmente para entender qué produce la tokenización antes de delegar el trabajo a `CountVectorizer`. El `TweetTokenizer` de NLTK maneja bien textos cortos e informales, emoticones, hashtags y signos de puntuación.

**Nota de implementación:** la pertenencia al vocabulario se comprueba contra un `set`, lo que da una búsqueda promedio O(1) en lugar del costo O(n) de `if word not in list`.


In [8]:
# Tokenizador de NLTK pensado para textos cortos e informales (conserva emoticones y puntuación)
tknzr = TweetTokenizer()
vocab_set = set()   # un set evita duplicados y permite búsquedas O(1)

for row in corpus['Reseña'].tolist():
    for word in tknzr.tokenize(row):
        vocab_set.add(word.lower())   # normaliza a minúsculas

bag = sorted(vocab_set)   # ordena para obtener un vocabulario reproducible
print(f'Tamaño del vocabulario: {len(bag)}')
print(f'Primeros 20 tokens: {bag[:20]}')

Tamaño del vocabulario: 88
Primeros 20 tokens: [',', '.', 'a', 'aburrida', 'actuaciones', 'atención', 'auténticas', 'bien', 'brillantes', 'cada', 'captar', 'carece', 'cine', 'cinematográfica', 'coherencia', 'completa', 'con', 'confusa', 'conmovedora', 'construidos']


> **Interpretación de la salida:**
>
> - ~88 tokens únicos en minúsculas a partir de 200 reseñas cortas.
> - Se incluyen **tokens de puntuación** (`.`, `,`): `TweetTokenizer` los conserva. Un *pipeline* de producción los filtraría con una longitud mínima de caracteres o una lista de *stopwords*.
> - `sorted(vocab_set)` garantiza un **orden reproducible** entre ejecuciones, ya que los conjuntos en Python no tienen orden.
> - Usar un `set` evita el costo O(n²) de `if word not in bag` sobre una lista creciente, algo crítico en corpus grandes.


---

## 4. Bag-of-Words con CountVectorizer

`CountVectorizer` construye el vocabulario y la matriz documento–término en una sola llamada a `fit_transform`:
- **`fit`** — recorre todos los documentos y construye un vocabulario interno (ordenado y en minúsculas)
- **`transform`** — convierte cada documento en un vector de conteos sobre ese vocabulario

El resultado es una **`csr_matrix` dispersa** de dimensión `(n_documentos, |V|)`.


In [9]:
# analyzer='word' cuenta palabras; ngram_range=(1, 1) usa solo unigramas
bow = CountVectorizer(analyzer='word', ngram_range=(1, 1))
features = bow.fit_transform(corpus['Reseña'].tolist())   # fit: vocabulario · transform: conteos

print(f'Dimensión de la matriz de características: {features.shape}')
print(f'Tamaño del vocabulario (CountVectorizer): {len(bow.vocabulary_)}')
# Dispersión = proporción de celdas en cero
print(f'Dispersión: {1 - features.nnz / (features.shape[0] * features.shape[1]):.2%}')

Dimensión de la matriz de características: (200, 84)
Tamaño del vocabulario (CountVectorizer): 84
Dispersión: 93.37%


> **Interpretación de la salida:**
>
> - La **dimensión** es `(200, |V|)`: una fila por reseña y una columna por término del vocabulario.
> - El **vocabulario de CountVectorizer** puede diferir del conteo manual porque usa su propio tokenizador basado en expresiones regulares y, por defecto, pasa a minúsculas sin eliminar tildes.
> - La **dispersión** supera el 95 % en documentos cortos: la mayoría de las palabras del vocabulario no aparecen en una reseña dada. Esto motiva la ponderación TF-IDF (reduce el dominio de las palabras comunes) o los *embeddings* densos (capturan similitud semántica).


In [10]:
# Vista densa — práctica solo para matrices de características pequeñas
features.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       [1, 0, 0, ..., 0, 1, 0]], shape=(200, 84))

---

## 5. Matriz de características como DataFrame


In [12]:
# Convierte la matriz dispersa en un DataFrame legible: filas = reseñas, columnas = palabras
df = pd.DataFrame(
    features.toarray(),
    index=[f'#{i}' for i in range(1, 1 + len(corpus))],
    columns=bow.get_feature_names_out(),
)
df.index.name = 'Reseña'
df[:10]

,aburrida,actuaciones,atención,auténticas,bien,brillantes,cada,captar,carece,cine,cinematográfica,coherencia,completa,con,confusa,conmovedora,construidos,contada,contemporáneo,de,decepcionante,dejan,del,desear,dirección,efectos,emoción,emotiva,en,encantó,entretenida,escena,esta,están,excelente,experiencia,gran,guion,historia,hizo,imágenes,inolvidable,la,las,lento,lograda,logró,los,maestra,mal,maravillosas,me,mi,momento,mucho,muy,música,nada,nadie,narrativa,ningún,no,obra,película,personajes,pobre,predecible,pérdida,pésimas,que,recomendada,recomendaría,reflexionar,rescatable,ritmo,simplemente,sin,son,sólido,tiempo,trama,un,una,visuales
Reseña,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
#1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
#2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
#3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0
#4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
#5,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
#6,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0
#7,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
#8,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0
#9,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


> **Interpretación de la salida:**
>
> - Cada **fila** es una reseña; cada **columna** es un término del vocabulario; cada **celda** es el conteo bruto de ocurrencias.
> - La mayoría de las celdas son **cero**, lo que confirma la alta dispersión. En la práctica, conviene almacenarla como matriz dispersa (`features`) en lugar de un `DataFrame` denso.
> - Reseñas idénticas producen vectores de conteo idénticos: BoW no distingue duplicados ni paráfrasis.

## Resumen

| Concepto | Detalle |
|---------|--------|
| Representación | Vector de conteos por documento |
| Orden de las palabras | **Se pierde** — BoW es independiente de la posición |
| Vocabulario | Se construye a partir del corpus de entrenamiento en `fit` |
| Tipo de matriz | Dispersa (`scipy.sparse.csr_matrix`) |
| Limitación principal | Alta dispersión; no captura similitud semántica |
| Siguiente paso | Ponderar los conteos por la frecuencia inversa de documento → **TF-IDF** |
